# Trabajo final · Girar un espín con un campo eléctrico

**Métodos Numéricos · UNAL Manizales · octubre–noviembre de 2026 · Samuel Camilo Merchán**

En tu parcial demostraste la regla de Gauss–Hermite integrando el osculante y la usaste en el pozo de Morse de H³⁵Cl. Este trabajo lleva esa misma herramienta a un problema de espín: **la resonancia de espín por dipolo eléctrico (EDSR)**. Un electrón está en un punto cuántico casi armónico y tiene acoplamiento espín–órbita. Un campo eléctrico oscilante lo sacude y le voltea el espín. Vas a calcular la frecuencia de Rabi de tres maneras, a descubrir cuál de ellas miente sobre el principio variacional y a **certificar** cuántos nodos bastan.

| Pieza | Semana | Celdas |
|---|---|---|
| A · Del árbol al DAG, y el modelo | 1 | A1–A4 |
| B · Tres discretizaciones | 2 | B1–B3 |
| C · El certificado y el mapa | 3 | C1–C6 |
| E · ¿Dónde vive la naturaleza? (opcional) | 4 | E1 |

Ritual de siempre: **predice en tu guía → corre la celda → explica la diferencia**. Solo cambias lo marcado con `# <-- CAMBIA ESTO`.

### Unidades y modelo
Usamos ħ = m* = ω = 1. Las longitudes van en unidades de ℓ = √(ħ/(m*ω)) y las energías en unidades de ħω. El hamiltoniano, en el **marco del laboratorio**, es

$$H = \tfrac12 p^2 + \tfrac12 x^2 + \eta\, p\,\sigma_y + \tfrac{b}{2}\,\sigma_z ,$$

donde η = ℓ/ℓ_so mide el espín–órbita (Rashba) y b = |g|μ_B B/ħω mide el Zeeman. El campo eléctrico entra como e·E₀·ℓ·x·cos(ω_d t). La frecuencia de Rabi es proporcional a |⟨g|x|e⟩|, el elemento de matriz de x entre los dos niveles más bajos.

### Fuente de los datos
- **Constantes físicas:** CODATA 2022 a través de `scipy.constants`. La celda 0 intenta además leer μ_B y ħ en la página de NIST (https://physics.nist.gov/cuu/Constants/) y compararlos.
- **Parámetros de los dispositivos:** copiados textualmente de los artículos y congelados en la celda 0, con su cita al lado. Los números del punto de GaAs son de Nowack et al., *Science* **318**, 1430 (2007), y los del nanoalambre de InAs, de Nadj-Perge et al., *Nature* **468**, 1084 (2010). La longitud de espín–órbita de InAs es de Fasth et al., *PRL* **98**, 266801 (2007), y las masas efectivas, de Vurgaftman, Meyer y Ram-Mohan, *J. Appl. Phys.* **89**, 5815 (2001), tablas I y III.
- **H³⁵Cl:** NIST Chemistry WebBook, igual que en tu parcial (celda B1). Fecha de consulta y de la copia congelada: **27 de octubre de 2026**.

In [ ]:
# Celda 0 · Constantes (CODATA, comprobadas contra NIST en línea) y parámetros congelados de los artículos
import numpy as np, pandas as pd, matplotlib.pyplot as plt, urllib.request, re
import scipy.constants as sc
from numpy.polynomial.hermite import hermgauss

hbar, h, e, me, muB = sc.hbar, sc.h, sc.e, sc.m_e, sc.physical_constants["Bohr magneton"][0]
origen = "scipy.constants (CODATA 2022)"
try:
    pag = urllib.request.urlopen(urllib.request.Request(
        "https://physics.nist.gov/cgi-bin/cuu/Value?mub", headers={"User-Agent": "Mozilla/5.0"}), timeout=10).read().decode("utf-8", "ignore")
    if "9.274 010 0657" in pag:
        origen += " · μ_B coincide con NIST en línea"
    else:
        origen += " · OJO: la página de NIST no muestra el mismo μ_B"
except Exception as err:
    origen += f" · NIST en línea no respondió ({type(err).__name__}); uso solo CODATA de scipy"
print("Constantes:", origen)

# Parámetros congelados (textuales de cada artículo). Unidades: meV, GHz, nm, m_e.
DISPOSITIVOS = pd.DataFrame([
    dict(nombre="GaAs (Nowack 2007)", Delta_meV=0.9, g=0.39, f_GHz=15.2, l_so_nm=32500.0, m_rel=0.067,
         fuente_l_so="Nowack et al. 2007: l_SO = 28–37 µm (usamos el punto medio)"),
    dict(nombre="InAs (Nadj-Perge 2010)", Delta_meV=7.5, g=9.2, f_GHz=13.0, l_so_nm=127.0, m_rel=0.026,
         fuente_l_so="Fasth et al. 2007, OTRO dispositivo de InAs: λ_SO ≈ 127 nm"),
]).set_index("nombre")
meV = 1e-3 * e
D = DISPOSITIVOS
D["ell_nm"] = hbar / np.sqrt(D.m_rel * me * D.Delta_meV * meV) * 1e9      # ℓ = ħ/√(m*Δ)
D["eta"] = D.ell_nm / D.l_so_nm                                            # η = ℓ/ℓ_so
D["b"] = h * D.f_GHz * 1e9 / (D.Delta_meV * meV)                          # b = hf/Δ (en resonancia)
print(D[["Delta_meV", "g", "f_GHz", "m_rel", "ell_nm", "eta", "b"]].round(5).to_string())

**Celda 1 · Herramientas.** No las modifiques, pero léelas: son las fórmulas de la guía.

- `ho(N)`: las matrices X, P y H₀ = diag(n + ½) en los N primeros estados del oscilador.
- `H_lab(N, eta, b)`: el hamiltoniano del **marco del laboratorio**. Todo es polinomial en x y p, así que sus elementos de matriz son exactos: el único error es el de truncar a N estados (Ritz).
- `H_rot(N, eta, b, modo)`: el **marco rotado**, H′ = U H U† con U = e^{iηxσ_y}. Queda H′ = H₀ − η²/2 + (b/2)[σ_z cos 2ηx − σ_x sin 2ηx]. El potencial deja de ser polinomial y hay dos maneras de llenar su matriz:
  - `modo="fbr"`: elementos de matriz exactos, calculados con una regla de 400 nodos;
  - `modo="dvr"`: la **misma** regla de N nodos de Gauss–Hermite. Es la DVR de Light, Hamilton y Lill, la de tu IV.1.
- `niveles(H, Xf)`: E₀, E₁, el desdoblamiento E₁ − E₀ y |⟨g|x|e⟩|.
- `residuo_lab(N, eta, b)`: la norma del residuo de los vectores de Ritz en el marco del laboratorio, **con la fórmula cerrada de la pieza C**.
- `encierro(N, eta, b)`: las cotas verificables de la pieza C para E₀, E₁, el desdoblamiento y |⟨g|x|e⟩|. Si las condiciones comprobables no se cumplen con ese N, devuelve `None`.
- `certificado_minimo(eta, b, tol)`: el N más pequeño con el que el encierro tiene ancho relativo ≤ tol.

In [ ]:
# Celda 1 · Herramientas
sx = np.array([[0, 1], [1, 0]], complex); sy = np.array([[0, -1j], [1j, 0]]); sz = np.array([[1, 0], [0, -1]], complex)

def ho(N):
    a = np.diag(np.sqrt(np.arange(1, N)), 1)
    return (a + a.T) / np.sqrt(2), 1j * (a.T - a) / np.sqrt(2), np.diag(np.arange(N) + 0.5)

def H_lab(N, eta, b):
    X, P, H0 = ho(N)
    return np.kron(H0, np.eye(2)) + eta * np.kron(P, sy) + b / 2 * np.kron(np.eye(N), sz), np.kron(X, np.eye(2))

_X400, _, _ = ho(400); _x400, _T400 = np.linalg.eigh(_X400)        # regla fina para la FBR
def H_rot(N, eta, b, modo="dvr"):
    X, P, H0 = ho(N)
    if modo == "dvr":
        xj, T = np.linalg.eigh(X)                                     # nodos de Gauss–Hermite (raíces de H_N)
        Cm, Sm = T @ np.diag(np.cos(2 * eta * xj)) @ T.T, T @ np.diag(np.sin(2 * eta * xj)) @ T.T
    else:
        Cm = (_T400 @ np.diag(np.cos(2 * eta * _x400)) @ _T400.T)[:N, :N]
        Sm = (_T400 @ np.diag(np.sin(2 * eta * _x400)) @ _T400.T)[:N, :N]
    H = np.kron(H0 - eta**2 / 2 * np.eye(N), np.eye(2)) + b / 2 * (np.kron(Cm, sz) - np.kron(Sm, sx))
    return H, np.kron(X, np.eye(2))

def niveles(H, Xf):
    E, V = np.linalg.eigh(H)
    return dict(E0=E[0], E1=E[1], dZ=E[1] - E[0], xge=abs(V[:, 0].conj() @ Xf @ V[:, 1]))

def residuo_lab(N, eta, b, k=3):
    '''‖r_k‖ = η √(N/2) ‖v_k(N−1)‖ para los k vectores de Ritz más bajos (pieza C, rompecabezas C.1).'''
    E, V = np.linalg.eigh(H_lab(N, eta, b)[0])
    return E[:k], eta * np.sqrt(N / 2) * np.linalg.norm(V[-2:, :k], axis=0)

def encierro(N, eta, b):
    '''Cotas verificables para E0, E1, el desdoblamiento y |<g|x|e>| (pieza C).
    Devuelve None si las condiciones comprobables no se cumplen con este N.'''
    E, V = np.linalg.eigh(H_lab(N, eta, b)[0])
    lam = E[:2]; r = eta * np.sqrt(N / 2) * np.linalg.norm(V[-2:, :2], axis=0)   # residuo cerrado (C.1)
    beta2 = 1.5 - eta**2 / 2 - b / 2                                            # Weyl en el marco rotado (C.3)
    if not (b < 1 and lam[1] - r[1] > lam[0] and lam[1] + r[1] < beta2):        # condiciones comprobables
        return None
    beta1 = lam[1] - r[1]                                                       # cota inferior de E1 (C.3)
    KT0 = r[0]**2 / (beta1 - lam[0]); KT1 = r[1]**2 / (beta2 - lam[1])         # Kato–Temple (C.2)
    d0, d1 = beta1 - lam[0], min(lam[1] - lam[0], beta2 - lam[1])               # brechas
    s0, s1 = r[0] / d0, r[1] / d1                                               # Davis–Kahan (C.4)
    if s0 >= 1 or s1 >= 1:
        return None
    Xf = np.kron(ho(N)[0], np.eye(2))
    X = abs(V[:, 0].conj() @ Xf @ V[:, 1])
    xu1 = np.sqrt(2 * (lam[1] + eta**2 / 2 + b / 2))                            # cota de momento (C.4)
    xv0 = np.linalg.norm(Xf @ V[:, 0])
    B = np.sqrt(2) * (s0 * xu1 + s1 * xv0)
    return dict(N=N, lam0=lam[0], lam1=lam[1], r0=r[0], r1=r[1], KT0=KT0, KT1=KT1,
                E0=(lam[0] - KT0, lam[0]), E1=(lam[1] - KT1, lam[1]),
                dZ=(lam[1] - KT1 - lam[0], lam[1] - lam[0] + KT0), X=(X - B, X + B), Xn=X, B=B)

def certificado_minimo(eta, b, tol=1e-6, Nmin=2, Nmax=80):
    '''El N más pequeño cuyo encierro tiene ancho relativo <= tol en |<g|x|e>| y en el desdoblamiento.'''
    for N in range(Nmin, Nmax + 1):
        c = encierro(N, eta, b)
        if c and c["B"] <= tol * c["Xn"] and (c["dZ"][1] - c["dZ"][0]) <= tol * c["dZ"][0]:
            return c
    return None

REF = {}
def referencia(eta, b):
    '''"Verdad": marco del laboratorio con 300 estados (se guarda para no repetir la cuenta).'''
    k = (round(eta, 12), round(b, 12))
    if k not in REF: REF[k] = niveles(*H_lab(300, eta, b))
    return REF[k]
print("Herramientas listas. Prueba: con η = 0 el elemento de Rabi es", niveles(*H_lab(10, 0.0, 0.2))["xge"])

---
## Pieza A · Del árbol al DAG, y el modelo (semana 1)

### Celda A1 · La transformación que se come al espín–órbita
La guía (A.2) te pide demostrar que U = e^{iηxσ_y} cambia H por H′ = H₀ − η²/2 + (b/2)[σ_z cos 2ηx − σ_x sin 2ηx]. Si es cierto, **los dos marcos tienen el mismo espectro** y el mismo |⟨g|x|e⟩|, porque x conmuta con U. **Predice:** con 300 estados en cada marco, ¿cuántos dígitos coinciden?

In [ ]:
# Celda A1 · Mismo espectro en los dos marcos
eta, b = 0.7, 0.2          # <-- CAMBIA ESTO
L = niveles(*H_lab(300, eta, b)); R = niveles(*H_rot(300, eta, b, "fbr"))
for k in L:
    print(f"{k:4s}  laboratorio = {L[k]: .15f}   rotado = {R[k]: .15f}   diferencia = {abs(L[k] - R[k]):.1e}")
print("\nTres niveles más bajos de H_lab:", np.round(np.linalg.eigvalsh(H_lab(300, eta, b)[0])[:3], 12))
print("Sin Zeeman (b = 0) deberían ser 1/2 − η²/2, doble:", 0.5 - eta**2 / 2)

### Celda A2 · Tu regla de Gauss–Hermite, otra vez
La cuenta a mano A.3 te pide ⟨1| sin 2ηx |0⟩ = √2·η·e^{−η²}. Aquí la calculamos con **tu** función del parcial: ⟨1|f|0⟩ = (1/√π) ∫ e^{−x²} √2·x·f(x) dx, con n nodos. **Predice:** sin 2ηx no es un polinomio. ¿La regla da el valor exacto con algún n finito? ¿Cómo baja el error con n, y depende de η?

In [ ]:
# Celda A2 · ⟨1|sin 2ηx|0⟩ con la regla de Gauss–Hermite de tu parcial
def valor_esperado_GH(g, n):                     # la de tu parcial (Celda 3)
    x, w = hermgauss(n)
    return np.sum(w * g(x)) / np.sqrt(np.pi)

ETAS = [0.1, 0.7, 1.5]   # <-- CAMBIA ESTO
filas = []
for n in [1, 2, 3, 4, 6, 8, 10, 14]:
    fila = {"n": n}
    for eta in ETAS:
        exacto = np.sqrt(2) * eta * np.exp(-eta**2)
        regla = valor_esperado_GH(lambda x: np.sqrt(2) * x * np.sin(2 * eta * x), n)
        fila[f"error η={eta}"] = abs(regla - exacto)
    filas.append(fila)
pd.set_option("display.float_format", lambda v: f"{v:.2e}")
print(pd.DataFrame(filas).to_string(index=False))

### Celda A3 · La fórmula de primer orden contra la cuenta completa
A primer orden en b, la guía (A.3) da |⟨g|x|e⟩| ≈ η·b·e^{−η²}. Sin el factor e^{−η²}, esta es la ecuación 2 de Nowack et al. (2007), que ellos atribuyen a Golovach, Borhani y Loss (2006). **Predice:** ¿dónde falla la fórmula en más de un 1 %, para η grande o para b grande? ¿Qué importa más, el factor e^{−η²} o el orden siguiente en b?

In [ ]:
# Celda A3 · Primer orden vs. diagonalización (300 estados)
filas = []
for eta in [0.05, 0.1, 0.3, 0.7, 1.0]:        # <-- CAMBIA ESTO
    for b in [0.01, 0.1, 0.3, 0.5]:
        x = referencia(eta, b)["xge"]
        f1 = eta * b * np.exp(-eta**2); f0 = eta * b
        filas.append(dict(eta=eta, b=b, xge=x, err_rel_con_exp=abs(f1 - x) / x, err_rel_sin_exp=abs(f0 - x) / x))
T = pd.DataFrame(filas)
print(T.to_string(index=False))
print("\nmáximo de η·e^{−η²}: η =", round(1 / np.sqrt(2), 4))

### Celda A4 · Los dos dispositivos reales
Con los parámetros de la celda 0 calculamos η, b y la frecuencia de Rabi **por cada kV/m de campo eléctrico** en el punto. Los artículos no reportan el campo absoluto, así que no podemos predecir el 4,7 MHz de Nowack. Sí podemos preguntar qué campo haría falta para obtenerlo. **Predice:** ¿en cuál de los dos dispositivos falla más la fórmula de primer orden?

In [ ]:
# Celda A4 · GaAs e InAs
E0 = 1e3                                         # V/m  <-- CAMBIA ESTO (campo de prueba)
for nombre, d in D.iterrows():
    r = referencia(d.eta, d.b)
    f1 = d.eta * d.b * np.exp(-d.eta**2)
    fR = e * E0 * d.ell_nm * 1e-9 * r["xge"] / h          # f_Rabi = e E0 ℓ |⟨g|x|e⟩| / h
    print(f"{nombre:24s} η = {d.eta:.2e}  b = {d.b:.4f}  |⟨g|x|e⟩| = {r['xge']:.4e}  "
          f"error de primer orden = {abs(f1 - r['xge']) / r['xge']:.1e}  f_Rabi = {fR/1e6:.3f} MHz por kV/m")
d = D.loc["GaAs (Nowack 2007)"]; r = referencia(d.eta, d.b)
print(f"\nCampo que haría falta en GaAs para 4,7 MHz: {4.7e6 * h / (e * d.ell_nm * 1e-9 * r['xge']) / 1e3:.1f} kV/m")

---
## Pieza B · Tres discretizaciones (semana 2)

### Celda B1 · La DVR en tu molécula
Antes del espín, volvemos a tu HCl. Resolvemos Morse con N estados del oscilador de dos maneras: con elementos de matriz exactos (FBR, que es Ritz) y con la DVR (el potencial evaluado en los N nodos de Gauss–Hermite, como en tu IV.1). Morse tiene niveles exactos E_v = (v + ½) − (ωₑxₑ/ωₑ)(v + ½)² en unidades de ħω. **Predice:** ¿la DVR queda por encima o por debajo de los niveles exactos?

In [ ]:
# Celda B1 · Morse de H35Cl con DVR y FBR (constantes de NIST, como en tu parcial)
URL_NIST = "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7647010&Mask=1000"
we, wexe = 2990.9463, 52.8186                    # copia congelada (NIST, 15 oct 2026)
try:
    html = urllib.request.urlopen(urllib.request.Request(URL_NIST, headers={"User-Agent": "Mozilla/5.0"}), timeout=15).read().decode("utf-8", "ignore")
    print("NIST en línea:", "coinciden" if ("2990.9463" in html and "52.8186" in html) else "OJO, no coinciden")
except Exception as err:
    print("NIST no respondió (", type(err).__name__, ") -> uso la copia congelada")
Dm = we / (4 * wexe); am = 1 / np.sqrt(2 * Dm)
Vg = lambda x: Dm * (1 - np.exp(-am * x))**2 - x**2 / 2          # V_Morse − V_arm (tu g del parcial)
E_exacto = lambda v: (v + 0.5) - wexe / we * (v + 0.5)**2

def H_morse(N, modo):
    X, P, H0 = ho(N)
    if modo == "dvr":
        xj, T = np.linalg.eigh(X); return H0 + T @ np.diag(Vg(xj)) @ T.T
    return H0 + (_T400 @ np.diag(Vg(_x400)) @ _T400.T)[:N, :N]

filas = []
for N in [2, 3, 4, 6, 8, 10, 12, 15, 20]:         # <-- CAMBIA ESTO
    d_, f_ = np.linalg.eigvalsh(H_morse(N, "dvr"))[:2], np.linalg.eigvalsh(H_morse(N, "fbr"))[:2]
    filas.append(dict(N=N, DVR_E0=(d_[0] - E_exacto(0)) * we, FBR_E0=(f_[0] - E_exacto(0)) * we,
                      DVR_E1=(d_[1] - E_exacto(1)) * we, FBR_E1=(f_[1] - E_exacto(1)) * we))
print("errores con signo, en cm⁻¹ (aproximado − exacto):")
print(pd.DataFrame(filas).to_string(index=False))
# Conclusión de la guía (B.2): "en los 18 casos la DVR queda por encima, así que la DVR, como Ritz, da cotas superiores."

### Celda B2 · Las tres discretizaciones del EDSR
Ahora el espín. Para cada N comparamos los errores **con signo** de E₀ y el error relativo de |⟨g|x|e⟩| en las tres discretizaciones:
- **laboratorio:** todo polinomial, solo error de truncamiento;
- **rotado, FBR:** potencial oscilante con elementos de matriz exactos;
- **rotado, DVR:** potencial oscilante evaluado en N nodos.

**Predice en tu guía** el signo de cada columna de E₀ antes de correr.

In [ ]:
# Celda B2 · Errores con signo
eta, b = 1.5, 0.2          # <-- CAMBIA ESTO (la guía te pide también η = 0,3)
ref = referencia(eta, b)
filas = []
for N in [2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24]:
    o = {"lab": niveles(*H_lab(N, eta, b)), "FBR": niveles(*H_rot(N, eta, b, "fbr")), "DVR": niveles(*H_rot(N, eta, b, "dvr"))}
    fila = {"N": N}
    for k, v in o.items():
        fila[f"{k}: E0−ref"] = v["E0"] - ref["E0"]
        fila[f"{k}: x rel"] = abs(v["xge"] - ref["xge"]) / ref["xge"]
    filas.append(fila)
pd.set_option("display.float_format", lambda v: f"{v:+.2e}")
print(f"η = {eta}, b = {b}:  E0 = {ref['E0']:.12f}  desdoblamiento = {ref['dZ']:.12f}  |⟨g|x|e⟩| = {ref['xge']:.12f}\n")
print(pd.DataFrame(filas).to_string(index=False))
print("\nCasos con E0 POR DEBAJO del valor de referencia:",
      {k: sum(f[f'{k}: E0−ref'] < -1e-13 for f in filas) for k in ["lab", "FBR", "DVR"]})

### Celda B3 · ¿Cuál marco gana?
Graficamos el error de E₀ contra N para η = 0,3 y η = 1,5. **Predice:** ¿el marco rotado, que tiene el espín–órbita eliminado de manera exacta, converge siempre más rápido que el del laboratorio?

In [ ]:
# Celda B3 · Convergencia en los tres esquemas
fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
Ns = np.arange(2, 31)
for ax, eta in zip(ejes, [0.3, 1.5]):           # <-- CAMBIA ESTO
    b = 0.2; ref = referencia(eta, b)
    for nombre, f, est in [("laboratorio (Ritz)", lambda N: H_lab(N, eta, b), "o-"),
                           ("rotado, FBR", lambda N: H_rot(N, eta, b, "fbr"), "s--"),
                           ("rotado, DVR", lambda N: H_rot(N, eta, b, "dvr"), "x:")]:
        ax.semilogy(Ns, [abs(niveles(*f(N))["E0"] - ref["E0"]) + 1e-17 for N in Ns], est, label=nombre, ms=4)
    ax.axhline(np.finfo(float).eps, color="gray", ls=":")
    ax.set_title(f"η = {eta}, b = {b}"); ax.set_xlabel("N (estados del oscilador)")
ejes[0].set_ylabel("|E0 − E0 ref|"); ejes[0].legend(); plt.tight_layout()
plt.savefig("fig_convergencia.pdf", bbox_inches="tight"); plt.show()

---
## Pieza C · El certificado y el mapa (semana 3)

### Celda C1 · El residuo tiene forma cerrada
En el marco del laboratorio, H saca al vector de Ritz v del espacio de N estados solo a través del término η p σ_y, y solo desde el nivel N − 1. La guía (C.1) te pide demostrar que **‖Hv − λv‖ = η √(N/2) ‖v_{N−1}‖**. Aquí lo comprobamos calculando Hv − λv directamente en N + 1 estados. **Predice:** ¿qué tan bien coinciden las dos columnas?

In [ ]:
# Celda C1 · Fórmula cerrada vs. cálculo directo
eta, b = 0.7, 0.2          # <-- CAMBIA ESTO
for N in [2, 4, 8, 12]:
    lam, r = residuo_lab(N, eta, b)
    E, V = np.linalg.eigh(H_lab(N, eta, b)[0]); Hg = H_lab(N + 1, eta, b)[0]
    v = np.zeros(2 * (N + 1), complex); v[:2 * N] = V[:, 0]
    directo = np.linalg.norm(Hg @ v - E[0] * v)
    print(f"N = {N:2d}: fórmula = {r[0]:.15e}   directo = {directo:.15e}")

### Celda C2 · Lineal contra cuadrático
Comparamos el error real de E₀ con dos cotas: **Weinstein** (|λ − E| ≤ ‖r‖) y **Kato–Temple** (λ − E ≤ ‖r‖²/(β − λ)). **Predice:** ¿cuántas veces más grande que el error real es cada cota?

In [ ]:
# Celda C2 · Weinstein vs. Kato–Temple
eta, b = 0.3, 0.2          # <-- CAMBIA ESTO
ref = referencia(eta, b)
PISO = 1e-13   # por debajo de esto, el "error real" es ruido de redondeo de la propia referencia
filas = []
for N in [3, 4, 5, 6, 7, 8, 10]:
    c = encierro(N, eta, b)
    if c is None: filas.append(dict(N=N, error_real=np.nan)); continue
    err = c["lam0"] - ref["E0"]
    filas.append(dict(N=N, error_real=err, Weinstein=c["r0"], Kato_Temple=c["KT0"],
                      KT_sobre_error=c["KT0"] / err if err > PISO else np.nan,
                      se_cumple=(-PISO <= err <= c["KT0"] + PISO)))
pd.set_option("display.float_format", lambda v: f"{v:.2e}")
print(pd.DataFrame(filas).to_string(index=False))

### Celda C3 · Tu cuenta a mano, revisada
La guía (C.5) te pide hacer a mano el caso N = 1 con b = 0. El vector de Ritz es |0⟩ con cualquier espín, λ = 1/2, el residuo vale η/√2 y el valor exacto es 1/2 − η²/2. Con b = 0 el nivel base es doble, así que β es el nivel siguiente, 3/2 − η²/2. **Predice:** ¿qué tan cerca está la cota de Kato–Temple del error real?

In [ ]:
# Celda C3 · N = 1, b = 0
for eta in [0.1, 0.3, 0.7]:     # <-- CAMBIA ESTO
    lam = 0.5; r = eta / np.sqrt(2); E = 0.5 - eta**2 / 2
    beta = 1.5 - eta**2 / 2
    print(f"η = {eta}: error real = {lam - E:.6f}   Kato–Temple = {r**2 / (beta - lam):.6f}   cociente = {r**2 / (beta - lam) / (lam - E):.4f}")

### Celda C4 · El encierro completo, puesto a prueba
`encierro` arma las piezas C.1 a C.4: el residuo cerrado, la cota de Weyl para E₂, Kato–Temple para E₀ y E₁, y Davis–Kahan con la cota de momento para |⟨g|x|e⟩|. Ninguna usa la referencia de 300 estados. Aquí lo probamos en 35 puntos (η, b). Para cada uno buscamos el N más pequeño con ancho relativo ≤ 10⁻⁶ y miramos si la referencia cae **dentro** de las tres cajas.

**Predice:** ¿cuántos puntos fallan? ¿Cuántas veces más ancha que el error real es la caja de |⟨g|x|e⟩|? ¿Y la de la energía?

In [ ]:
# Celda C4 · ¿La verdad cae dentro de la caja?
TOL = 1e-6        # <-- CAMBIA ESTO
filas, fallas = [], 0
for eta in [0.01, 0.1, 0.3, 0.7, 1.0, 1.5, 2.0]:
    for b in [0.005, 0.05, 0.2, 0.5, 0.9]:
        c = certificado_minimo(eta, b, TOL); R = referencia(eta, b)
        dentro = (c["E0"][0] - 1e-13 <= R["E0"] <= c["E0"][1] + 1e-13 and
                  c["E1"][0] - 1e-13 <= R["E1"] <= c["E1"][1] + 1e-13 and
                  c["X"][0] - 1e-13 <= R["xge"] <= c["X"][1] + 1e-13)
        fallas += not dentro
        filas.append(dict(eta=eta, b=b, N=c["N"], ancho_rel_X=c["B"] / c["Xn"],
                          error_real_X=abs(c["Xn"] - R["xge"]) / R["xge"], dentro=dentro))
T = pd.DataFrame(filas)
pd.set_option("display.float_format", lambda v: f"{v:.1e}")
print(T.to_string(index=False))
print(f"\nFallas: {fallas} de {len(T)}.   N máximo usado: {T.N.max()}")

### Celda C5 · El mapa de validez (el resultado central)
Con `certificado_minimo` en una malla de 45 × 40 puntos (η, b) acotamos, **con garantía**, el error relativo de tres fórmulas de primer orden:
1. |⟨g|x|e⟩| ≈ η·b, la ecuación (2) de Nowack et al. (Golovach et al.);
2. |⟨g|x|e⟩| ≈ η·b·e^{−η²}, la de tu cuenta A.3;
3. desdoblamiento ≈ b·e^{−η²}, el Zeeman renormalizado.

Las estrellas son los dos dispositivos de la celda 0. El cuarto panel muestra cuántos estados pidió el certificado. **Predice antes de correr:** ¿dónde está la curva del 1 % de la fórmula 1? ¿Y la de la fórmula 2? ¿Cuál de las dos depende de b y cuál de η? Tarda unos 10 segundos.

In [ ]:
# Celda C5 · Mapa certificado
ETA = np.geomspace(1e-3, 2, 46); BB = np.geomspace(0.005, 0.9, 40)     # <-- CAMBIA ESTO
FG, FL, FZ, NN = (np.full((len(BB), len(ETA)), np.nan) for _ in range(4))
def peor(f, caja): return max(abs(f - caja[0]) / caja[0], abs(f - caja[1]) / caja[1])
for i, b in enumerate(BB):
    Ninicio = 2
    for j, eta in enumerate(ETA):
        c = certificado_minimo(eta, b, 1e-6, Nmin=Ninicio)
        FG[i, j] = peor(eta * b, c["X"]); FL[i, j] = peor(eta * b * np.exp(-eta**2), c["X"])
        FZ[i, j] = peor(b * np.exp(-eta**2), c["dZ"]); NN[i, j] = c["N"]
        Ninicio = max(2, c["N"] - 3)

fig, ejes = plt.subplots(2, 2, figsize=(12, 9.5), sharex=True, sharey=True); ejes = ejes.ravel()
titulos = [r"(a) $\eta b$ (Golovach; Nowack Eq. 2)", r"(b) $\eta b\,e^{-\eta^2}$", r"(c) splitting $\approx b\,e^{-\eta^2}$"]
for ax, F, t in zip(ejes[:3], [FG, FL, FZ], titulos):
    cs = ax.contourf(ETA, BB, np.log10(np.maximum(F, 1e-9)), levels=np.arange(-9, 1.01, 0.5), cmap="viridis")
    cs.set_edgecolor("face")
    cl = ax.contour(ETA, BB, F, levels=[0.01, 0.05, 0.1], colors="w", linewidths=1.2)
    ax.clabel(cl, fmt={0.01: "1%", 0.05: "5%", 0.1: "10%"}, fontsize=8)
    ax.set_title(t); fig.colorbar(cs, ax=ax, label="log10 certified relative error", ticks=range(-9, 1, 2))
cn = ejes[3].contourf(ETA, BB, NN, levels=np.arange(2, NN.max() + 2, 2), cmap="magma")
cn.set_edgecolor("face")
fig.colorbar(cn, ax=ejes[3], label="oscillator states N"); ejes[3].set_title("(d) cost of the certificate")
for ax in ejes:
    ax.set_xscale("log"); ax.set_yscale("log")
    for nombre, d in D.iterrows(): ax.plot(d.eta, d.b, "r*", ms=11)
for ax in ejes[2:]: ax.set_xlabel(r"$\eta=\ell/\ell_{\rm so}$")
for ax in ejes[::2]: ax.set_ylabel(r"$b=|g|\mu_B B/\hbar\omega$")
plt.savefig("fig_mapa.pdf", bbox_inches="tight"); plt.show()

for nombre, d in D.iterrows():
    c = certificado_minimo(d.eta, d.b)
    print(f"{nombre:24s} error certificado:  η·b → {peor(d.eta * d.b, c['X']):.2e}   "
          f"η·b·e^(−η²) → {peor(d.eta * d.b * np.exp(-d.eta**2), c['X']):.2e}   N = {c['N']}")
j1 = np.argmax(FG[0] > 0.01); j10 = np.argmax(FG[0] > 0.1)
print(f"\nCon b = {BB[0]}: la fórmula η·b pasa del 1 % en η ≈ {ETA[j1]:.3f} y del 10 % en η ≈ {ETA[j10]:.3f}")
print(f"Máximo error de η·b·e^(−η²) con b ≤ 0,02: {np.nanmax(FL[BB <= 0.02]):.2e}")

### Celda C6 · Dos agentes y una regla aprendida
Un agente escoge la discretización (`lab`, `FBR` o `DVR`) y el número N de estados (de 2 a 30) para cada punto (η, b). Comparamos tres políticas:
1. **Agente variacional.** Su recompensa es −(E₀ + c·N): «menor energía es mejor», menos un costo por estado. Es el principio variacional de tu IV.1, convertido en recompensa.
2. **Agente certificado.** Usa el N más pequeño cuyo encierro tiene ancho relativo ≤ 10⁻⁶. Solo el marco del laboratorio tiene certificado.
3. **N aprendido.** Se busca el N más pequeño que cumple la tolerancia en el error **real** en puntos de entrenamiento con η ≤ 0,5, y después se usa ese mismo N en puntos nuevos.

**Predice:** ¿qué discretización escoge el agente variacional? ¿El N aprendido sirve con η = 1,5?

In [ ]:
# Celda C6 · Agentes
COSTO, TOL = 1e-4, 1e-6        # <-- CAMBIA ESTO (costo por estado y tolerancia relativa)
OPC = {"lab": lambda N, e_, b_: H_lab(N, e_, b_), "FBR": lambda N, e_, b_: H_rot(N, e_, b_, "fbr"),
       "DVR": lambda N, e_, b_: H_rot(N, e_, b_, "dvr")}
NS = range(2, 31)

def agente_variacional(eta, b):
    return min(((niveles(*OPC[m](N, eta, b))["E0"] + COSTO * N, m, N) for m in OPC for N in NS))

def error_rel_X(N, eta, b):
    ref = referencia(eta, b); return abs(niveles(*H_lab(N, eta, b))["xge"] - ref["xge"]) / ref["xge"]

ENTRENO = [(e_, b_) for e_ in [0.05, 0.1, 0.2, 0.3, 0.5] for b_ in [0.05, 0.2]]
N_APRENDIDO = min(N for N in NS if all(error_rel_X(N, e_, b_) <= TOL for e_, b_ in ENTRENO))
print("N aprendido en entrenamiento (η ≤ 0,5):", N_APRENDIDO, "\n")

filas = []
for eta in [0.1, 0.3, 0.7, 1.0, 1.5, 2.0]:      # <-- CAMBIA ESTO
    b = 0.2; ref = referencia(eta, b)
    _, m, Nv = agente_variacional(eta, b); Ev = niveles(*OPC[m](Nv, eta, b))["E0"]
    c = certificado_minimo(eta, b, TOL)
    filas.append(dict(eta=eta, variacional=f"{m}, N={Nv}", E0_var_menos_ref=Ev - ref["E0"],
                      certificado_N=c["N"], err_X_cert=error_rel_X(c["N"], eta, b),
                      err_X_N_aprendido=error_rel_X(N_APRENDIDO, eta, b)))
pd.set_option("display.float_format", lambda v: f"{v:+.2e}")
print(pd.DataFrame(filas).to_string(index=False))

---
## Pieza E (opcional) · ¿Dónde vive la naturaleza?

### Celda E1 · Agrega tu material
Agrega a la tabla una fila con un dispositivo que encuentres tú (por ejemplo, un nanoalambre de InSb o un punto de huecos en germanio). Cada número debe venir con su fuente verificada en la guía. **Predice:** ¿cae en la zona donde la fórmula de primer orden basta, o en la zona donde hace falta diagonalizar con certificado?

In [ ]:
# Celda E1 · Tu dispositivo
MIO = dict(nombre="(tu material)", Delta_meV=np.nan, g=np.nan, f_GHz=np.nan, l_so_nm=np.nan, m_rel=np.nan)   # <-- CAMBIA ESTO
filas = list(D[["Delta_meV", "g", "f_GHz", "l_so_nm", "m_rel"]].reset_index().to_dict("records")) + [MIO]
for d in filas:
    if np.isnan(d["Delta_meV"]): print(f"{d['nombre']}: faltan datos"); continue
    ell = hbar / np.sqrt(d["m_rel"] * me * d["Delta_meV"] * meV) * 1e9
    eta, b = ell / d["l_so_nm"], h * d["f_GHz"] * 1e9 / (d["Delta_meV"] * meV)
    r = referencia(eta, b); f1 = eta * b * np.exp(-eta**2)
    print(f"{d['nombre']:24s} η = {eta:.3e}  b = {b:.4f}  error de primer orden = {abs(f1 - r['xge']) / r['xge']:.1e}  "
          f"N certificado = {(certificado_minimo(eta, b, Nmax=150) or {}).get('N')}")

---
### Antes de entregar
**Entorno de ejecución → Reiniciar y ejecutar todo.** Si alguna celda falla o una cifra cambia, arréglalo antes de copiarla al artículo. Solo el 4 % de los cuadernos públicos de GitHub reproduce sus propios resultados guardados (Pimentel et al., 2019). El tuyo tiene que estar en ese 4 %.